# 01 · Explore My Training

**Phase 1** of the project: what does my running actually look like, and how has my training load changed over time?

Before running this notebook, download your data:

```bash
python modules/get_data.py
```

That saves `data/00-raw/strava_activities.json`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from modules.training import load_runs, daily_load, training_load, weekly_summary, riegel, fmt_time, RACE_DISTANCES_M

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.precision", 2)

## 1. Load and clean runs

In [ ]:
runs = load_runs("data/00-raw/strava_activities.json")
runs.to_csv("data/02-processed/runs.csv", index=False)

print(f"{len(runs)} runs from {runs.date.min():%b %d, %Y} to {runs.date.max():%b %d, %Y}")
print(f"{runs.distance_km.sum():,.0f} km total · {runs.avg_hr.notna().mean():.0%} of runs have heart rate")
runs.tail()

## 2. Weekly volume

✏️ **Look for:** build-ups before races, drops (injury? exams? travel?), and how steady your long run is.

In [ ]:
weekly = weekly_summary(runs)

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(weekly.index, weekly.km, width=5, color="#4C72B0", label="weekly km")
ax.plot(weekly.index, weekly.long_run_km, color="#DD8452", marker=".", label="long run (km)")
ax.set(ylabel="km", title="Weekly running volume")
ax.legend(frameon=False)
plt.tight_layout(); plt.savefig("results/weekly_volume.png", dpi=150); plt.show()

## 3. Training load: fitness, fatigue, and injury risk

Each run gets a **load** score based on how long and how hard it was (heart-rate TRIMP, or minutes when heart rate is missing).

- **Fitness**: the long-term (42-day) average load
- **Fatigue**: the short-term (7-day) average load
- **Form**: fitness − fatigue. Negative means you're carrying fatigue; positive means you're fresh.
- **ACWR** (acute:chronic workload ratio): this week's load relative to the last 4 weeks. Spikes above about **1.5** are a commonly cited injury-risk warning.

In [ ]:
load = training_load(daily_load(runs))

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(load.index, load.fitness, label="fitness (42-day)", color="#4C72B0")
axes[0].plot(load.index, load.fatigue, label="fatigue (7-day)", color="#DD8452", alpha=0.8)
axes[0].set(ylabel="load", title="Fitness and fatigue")
axes[0].legend(frameon=False)

axes[1].plot(load.index, load.acwr, color="#55A868")
axes[1].axhspan(1.5, max(2, load.acwr.max()), color="#C44E52", alpha=0.12, label="injury-risk zone (> 1.5)")
axes[1].axhspan(0.8, 1.3, color="#55A868", alpha=0.10, label="'sweet spot' (0.8–1.3)")
axes[1].set(ylabel="ACWR", title="Acute:chronic workload ratio")
axes[1].legend(frameon=False, loc="upper left")
plt.tight_layout(); plt.savefig("results/training_load.png", dpi=150); plt.show()

print(f"Days in the risk zone: {load.risk_flag.sum()}")

## 4. Pace vs. heart rate: am I getting fitter?

If fitness improves, the same heart rate should produce a faster pace over time.

In [ ]:
hr = runs.dropna(subset=["avg_hr"])
fig, ax = plt.subplots(figsize=(7, 4.5))
sc = ax.scatter(hr.avg_hr, hr.pace_min_per_km, c=hr.date.map(pd.Timestamp.toordinal), cmap="viridis", s=14)
ax.invert_yaxis()
ax.set(xlabel="Average heart rate (bpm)", ylabel="Pace (min/km, faster ↑)", title="Pace vs. heart rate, colored by date")
cbar = fig.colorbar(sc, ax=ax)
cbar.set_ticks(cbar.get_ticks()[1:-1])
cbar.set_ticklabels([pd.Timestamp.fromordinal(int(t)).strftime("%b %Y") for t in cbar.get_ticks()])
plt.tight_layout(); plt.savefig("results/pace_vs_hr.png", dpi=150); plt.show()

## 5. Baseline race predictions (Riegel's formula)

Riegel: **T₂ = T₁ × (D₂ / D₁)^1.06**. This is the baseline the Phase 2 model has to beat.

✏️ Mark your races in Strava (Edit activity → Run type → **Race**) so they show up here. If there are none, the notebook uses your fastest run of at least 5 km.

In [ ]:
races = runs[runs.workout_type == "race"]
ref = races.iloc[-1] if len(races) else runs[runs.distance_km >= 5].nsmallest(1, "pace_min_per_km").iloc[0]
print(f"Reference: {ref['name']} on {ref.date:%b %d, %Y}: {ref.distance_km:.2f} km in {fmt_time(ref.moving_min)}\n")

pd.DataFrame({
    "predicted time": {name: fmt_time(riegel(ref.moving_min, ref.distance_km * 1000, d)) for name, d in RACE_DISTANCES_M.items()}
})

## ✏️ Notes for Phase 2

- What surprised you in your training history?
- Which features seem related to fitness (weekly km, long run, fitness score, pace at a given heart rate)?
- Is there enough race data to train on, or should "race performance" be estimated from hard efforts?